# Cubierta Forestal · Experimentación de Machine Learning (parte derecha de la arquitectura)

**Alcance de este notebook** — Jupyter + PostgreSQL + preparación y entrenamiento del modelo
+ 25 configuraciones de hiperparámetros + resultados listos para que **otro componente** los
registre en MLflow.

Flujo:

```
PostgreSQL → Validación de datos → Gold Layer → EDA → Feature Engineering
         → Train/Validation/Test → Modelo único (AutoGluon, una sola familia)
         → 25 configuraciones → 25 experimentos → Métricas
         → Tabla consolidada de resultados → Preparado para MLflow
```

**Fuera de alcance aquí** (otros integrantes lo integran después): MLflow, MinIO, FastAPI,
Model Registry, API de inferencia y el `docker-compose` completo. Este notebook **no** usa
ninguno de esos componentes; sólo deja los artefactos y la estructura conceptual listos.

> Este notebook es **nuevo** y no modifica el notebook existente
> `cubierta-forestal.ipynb` (flujo mínimo scikit-learn → MinIO). Tampoco modifica la
> fuente de datos en PostgreSQL: todo lo que escribe va al sistema de archivos local
> (`models/`, `results.csv`, `results.parquet`, `mlflow_payloads/`).

## 01. Configuración del entorno

Reproducibilidad primero: una única **semilla global** y una estructura de carpetas fija.
AutoGluon es la única dependencia pesada que puede faltar en el entorno base (el `pyproject.toml`
compartido con la *inference API* se deja intacto a propósito). La celda siguiente la instala
sólo si no está disponible, para que el notebook pueda ejecutarse *desde cero*.

In [ ]:
# --- Semilla global y rutas de salida (reproducibilidad) ---
from pathlib import Path

SEED = 42                      # semilla única usada en todo el notebook
N_CONFIGS = 25                 # exactamente 25 configuraciones / experimentos

# Raíz de artefactos: junto al notebook, sin tocar PostgreSQL ni MinIO.
ARTIFACTS_DIR = Path("artifacts").resolve()
MODELS_DIR = ARTIFACTS_DIR / "models"            # models/config_01 ... config_25
RESULTS_DIR = ARTIFACTS_DIR / "results"          # results.csv / results.parquet
MLFLOW_DIR = ARTIFACTS_DIR / "mlflow_payloads"   # un JSON por run, listo para MLflow
for d in (MODELS_DIR, RESULTS_DIR, MLFLOW_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("SEED      =", SEED)
print("N_CONFIGS =", N_CONFIGS)
print("artifacts ->", ARTIFACTS_DIR)

In [ ]:
# --- Bootstrap de dependencias que pueden faltar en el entorno base ---
# Idempotente: si ya están instaladas no hace nada. AutoGluon restringido a tabular+LightGBM
# para no arrastrar visión/NLP. python-dotenv, pyarrow, matplotlib y seaborn son opcionales.
import importlib
import subprocess
import sys


def _ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        return False
    except ImportError:
        print(f"instalando {pip_name} ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", pip_name]
        )
        return True


# Núcleo del experimento (obligatorio para entrenar):
_ensure("autogluon.tabular[lightgbm]", "autogluon.tabular")
# Utilidades (si alguna falla, el notebook degrada con elegancia más abajo):
for _pkg, _imp in [
    ("python-dotenv", "dotenv"),
    ("pyarrow", "pyarrow"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("psycopg[binary]", "psycopg"),
    ("SQLAlchemy", "sqlalchemy"),
]:
    try:
        _ensure(_pkg, _imp)
    except Exception as exc:  # noqa: BLE001
        print(f"aviso: no se pudo instalar {_pkg}: {exc}")

print("bootstrap listo")

## 02. Importación de librerías

In [ ]:
import io
import json
import os
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, inspect, text

# AutoGluon (familia única de modelo; ver sección 12)
from autogluon.tabular import TabularPredictor

# scikit-learn sólo para el cálculo explícito y controlado de métricas
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

np.random.seed(SEED)
print("librerías importadas")

## 03. Configuración de variables de entorno

**No se escriben credenciales en el notebook.** La conexión se arma a partir de variables de
entorno. En el contenedor `jupyter` de `docker-compose` ya vienen inyectadas
(`DATA_DB_HOST`, `DATA_DB_PORT`, `DATA_DB_NAME`, `DATA_DB_USER`, `DATA_DB_PASSWORD`). Para
ejecución local fuera de Docker se pueden definir en un archivo `.env` (nunca versionado):

```dotenv
DATA_DB_HOST=localhost
DATA_DB_PORT=5432
DATA_DB_NAME=cubierta_forestal
DATA_DB_USER=app
DATA_DB_PASSWORD=app
```

In [ ]:
# Carga opcional de .env (si python-dotenv está disponible y el archivo existe).
try:
    from dotenv import find_dotenv, load_dotenv

    _dotenv = find_dotenv(usecwd=True)
    if _dotenv:
        load_dotenv(_dotenv, override=False)
        print(f".env cargado desde {_dotenv}")
    else:
        print("no se encontró .env; se usan las variables de entorno del proceso")
except Exception as exc:  # noqa: BLE001
    print(f"python-dotenv no disponible ({exc}); se usan variables de entorno del proceso")

# Parámetros de conexión (con los valores por defecto del docker-compose del proyecto).
DB_CONFIG = {
    "host": os.environ.get("DATA_DB_HOST", "localhost"),
    "port": os.environ.get("DATA_DB_PORT", "5432"),
    "dbname": os.environ.get("DATA_DB_NAME", "cubierta_forestal"),
    "user": os.environ.get("DATA_DB_USER", "app"),
    # La contraseña nunca se imprime.
    "password": os.environ.get("DATA_DB_PASSWORD", "app"),
}
print({k: v for k, v in DB_CONFIG.items() if k != "password"})

## 04. Conexión a PostgreSQL

Función reutilizable `connect_to_postgresql()` construida sobre **SQLAlchemy** + **psycopg v3**
(el driver ya instalado en el proyecto; `psycopg2` sería intercambiable cambiando el prefijo del
DSN). Devuelve un `Engine` con `pool_pre_ping=True` y valida la conexión con un `SELECT 1`.

In [ ]:
def connect_to_postgresql(config: dict = DB_CONFIG, driver: str = "psycopg", echo: bool = False):
    '''Crea y valida un SQLAlchemy Engine contra la base del proyecto.

    Parameters
    ----------
    config : dict con host, port, dbname, user, password.
    driver : 'psycopg' (v3, por defecto) o 'psycopg2'. El resto del DSN es idéntico.
    echo   : si True, SQLAlchemy registra el SQL emitido.

    Returns
    -------
    sqlalchemy.engine.Engine ya verificado (lanza excepción si la conexión falla).
    '''
    url = (
        f"postgresql+{driver}://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['dbname']}"
    )
    engine = create_engine(url, pool_pre_ping=True, echo=echo)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    return engine


engine = connect_to_postgresql()
with engine.connect() as conn:
    server = conn.execute(text("SELECT version()")).scalar()
    db = conn.execute(text("SELECT current_database()")).scalar()
    usr = conn.execute(text("SELECT current_user")).scalar()
print("conexión OK")
print("base   :", db, "| usuario:", usr)
print("servidor:", server)

## 05. Exploración de la fuente de datos

Antes de asumir nada, se inspecciona el esquema real: esquemas, tablas, columnas, tipos y
conteos. La arquitectura del proyecto sigue un patrón *medallion*:

| Esquema | Tabla | Rol |
|---|---|---|
| `raw` | `covertype` | filas tal cual llegan de la Data API |
| `processed` | `covertype` | limpio, tipado, validado (CHECKs), deduplicado |
| `training` | `dataset_version` | metadatos de cada snapshot versionado |
| `training` | `covertype_features` | snapshot inmutable con split `train/validation/test` ya decidido |

In [ ]:
insp = inspect(engine)
schemas = [s for s in insp.get_schema_names() if s in ("raw", "processed", "training")]
print("esquemas del proyecto:", schemas)

inventory = []
for schema in schemas:
    for tbl in insp.get_table_names(schema=schema):
        with engine.connect() as conn:
            n = conn.execute(text(f'SELECT count(*) FROM "{schema}".{tbl}')).scalar()
        inventory.append({"schema": schema, "table": tbl, "rows": n})
inventory_df = pd.DataFrame(inventory)
inventory_df

In [ ]:
# Columnas y tipos de cada tabla relevante.
for schema, tbl in [("raw", "covertype"), ("processed", "covertype"),
                    ("training", "covertype_features"), ("training", "dataset_version")]:
    try:
        cols = insp.get_columns(tbl, schema=schema)
        print(f"\n=== {schema}.{tbl} ===")
        for c in cols:
            print(f"  {c['name']:<36} {str(c['type'])}")
    except Exception as exc:  # noqa: BLE001
        print(f"\n=== {schema}.{tbl} === (no disponible: {exc})")

## 06. Validación de calidad

Sobre `processed.covertype` (stage limpio) se verifica: nulos, únicos, duplicados del vector de
features y estadísticos descriptivos. Esto confirma qué garantiza ya la capa `processed` (CHECKs
de rango, normalización de texto, índice único sobre el vector de features) antes de construir la
Gold Layer analítica.

In [ ]:
PROCESSED_TABLE = "processed.covertype"

# Muestra acotada para perfilar sin traer toda la tabla si es muy grande.
with engine.connect() as conn:
    n_processed = conn.execute(text(f"SELECT count(*) FROM {PROCESSED_TABLE}")).scalar()
print(f"{PROCESSED_TABLE}: {n_processed:,} filas")

profile_df = pd.read_sql(text(f"SELECT * FROM {PROCESSED_TABLE}"), engine)
print("cargado para perfilado:", profile_df.shape)

In [ ]:
# Nulos, únicos y tipos por columna.
quality = pd.DataFrame({
    "dtype": profile_df.dtypes.astype(str),
    "n_null": profile_df.isna().sum(),
    "pct_null": (profile_df.isna().mean() * 100).round(3),
    "n_unique": profile_df.nunique(),
})
quality

In [ ]:
# Duplicados exactos sobre el vector de features del contrato (sin columnas de linaje).
FEATURE_COLS_DB = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
    "wilderness_area", "soil_type", "cover_type",
]
dups = profile_df.duplicated(subset=FEATURE_COLS_DB).sum()
print(f"duplicados exactos del vector de features: {dups} "
      f"(processed tiene índice único que debería forzar 0)")

# Estadísticos descriptivos (numéricos y categóricos por separado).
display(profile_df.describe(include=[np.number]).T)
display(profile_df.describe(include=["object"]).T)

## 07. Construcción / validación de Gold Layer

**Gold Layer = dataset analítico limpio, consistente, tipado y listo para ML.** No se asume que
una tabla cruda ya lo cumpla ni se modifica la fuente. Se define una **vista lógica** sobre
`training.covertype_features` (el snapshot versionado) que:

- selecciona sólo las columnas del contrato de features **+** la columna `split`,
- **excluye columnas de linaje** que provocarían *data leakage* o ruido de identificador
  (`id`, `processed_id`, `dataset_version`, timestamps, `dag_run_id`, `raw_id`),
- deja las categóricas como texto (la codificación pertenece al pipeline del modelo).

Trazabilidad: **Fuente** (`raw.covertype`) → **Transformaciones** (normalización + CHECKs en
`processed.covertype`, split determinista en `training.covertype_features`) → **Gold**
(vista `analytics.covertype_gold`) → **Dataset de ML** (DataFrame `gold_df`).

Si no existe aún un `dataset_version` congelado, el notebook degrada a `processed.covertype` y
crea un split determinista reproducible en memoria (sin tocar la base).

In [ ]:
# ¿Hay snapshots versionados disponibles?
try:
    versions = pd.read_sql(
        text("SELECT * FROM training.dataset_version ORDER BY created_at DESC"), engine
    )
except Exception as exc:  # noqa: BLE001
    versions = pd.DataFrame()
    print("training.dataset_version no disponible:", exc)

USE_DB_SPLIT = not versions.empty
print("versiones disponibles:", list(versions["version"]) if USE_DB_SPLIT else "ninguna")
versions

In [ ]:
# DDL de la Gold Layer como VISTA LÓGICA. Se intenta materializar en un esquema 'analytics'
# (solo lectura sobre la fuente). Si el usuario no tiene privilegios de DDL, no es un error:
# el flujo sigue leyendo el mismo SELECT directamente a un DataFrame.
GOLD_FEATURE_COLS = [c for c in FEATURE_COLS_DB]  # contrato de features + target
TARGET = "cover_type"

if USE_DB_SPLIT:
    DATASET_VERSION = versions.iloc[0]["version"]  # el más reciente; se puede fijar a mano
    gold_select = text(
        f"SELECT {', '.join(GOLD_FEATURE_COLS)}, split "
        f"FROM training.covertype_features WHERE dataset_version = :v"
    )
    gold_params = {"v": DATASET_VERSION}
    gold_view_sql = (
        "CREATE SCHEMA IF NOT EXISTS analytics;\n"
        "CREATE OR REPLACE VIEW analytics.covertype_gold AS\n"
        f"  SELECT {', '.join(GOLD_FEATURE_COLS)}, split\n"
        f"  FROM training.covertype_features\n"
        f"  WHERE dataset_version = '{DATASET_VERSION}';"
    )
else:
    DATASET_VERSION = "processed-fallback"
    gold_select = text(
        f"SELECT {', '.join(GOLD_FEATURE_COLS)} FROM {PROCESSED_TABLE}"
    )
    gold_params = {}
    gold_view_sql = (
        "CREATE SCHEMA IF NOT EXISTS analytics;\n"
        "CREATE OR REPLACE VIEW analytics.covertype_gold AS\n"
        f"  SELECT {', '.join(GOLD_FEATURE_COLS)} FROM {PROCESSED_TABLE};"
    )

print("DATASET_VERSION =", DATASET_VERSION)
print("\n-- Gold Layer (vista lógica) --\n" + gold_view_sql)

# Intento best-effort de materializar la vista (no obligatorio para el flujo).
try:
    with engine.begin() as conn:
        for stmt in gold_view_sql.split(";\n"):
            if stmt.strip():
                conn.execute(text(stmt))
    print("\nvista analytics.covertype_gold materializada")
except Exception as exc:  # noqa: BLE001
    print(f"\n(no se materializó la vista — se continúa leyendo el SELECT directamente: {exc})")

In [ ]:
# Carga del dataset Gold a memoria.
gold_df = pd.read_sql(gold_select, engine, params=gold_params)

if not USE_DB_SPLIT:
    # Split determinista reproducible EN MEMORIA (no se escribe en la base).
    # Hash estable por fila -> u in [0,1): 70% train, 15% validation, 15% test.
    import hashlib

    def _u(row_idx: int) -> float:
        h = hashlib.md5(f"{row_idx}:{SEED}".encode()).hexdigest()[:8]
        return int(h, 16) / 0xFFFFFFFF

    u = gold_df.index.to_series().map(_u)
    gold_df["split"] = np.where(u < 0.70, "train",
                        np.where(u < 0.85, "validation", "test"))

print("gold_df:", gold_df.shape)
display(gold_df["split"].value_counts())
gold_df.head()

In [ ]:
# --- Validaciones mínimas de la Gold Layer ---
checks = {}
checks["sin_nulos"] = bool(gold_df.drop(columns=["split"]).isna().sum().sum() == 0)
checks["target_presente"] = TARGET in gold_df.columns
checks["splits_esperados"] = set(gold_df["split"].unique()) <= {"train", "validation", "test"}

# Rangos (reflejan los CHECK de processed.covertype).
rng = {
    "aspect": (0, 360), "slope": (0, 90),
    "hillshade_9am": (0, 255), "hillshade_noon": (0, 255), "hillshade_3pm": (0, 255),
    "cover_type": (0, 6),
}
range_ok = True
for col, (lo, hi) in rng.items():
    if col in gold_df:
        in_range = gold_df[col].between(lo, hi).all()
        range_ok &= bool(in_range)
checks["rangos_validos"] = range_ok

# Distancias no negativas.
dist_cols = [c for c in gold_df.columns if c.startswith("horizontal_distance")]
checks["distancias_no_negativas"] = bool((gold_df[dist_cols] >= 0).all().all())

# Consistencia de categorías (texto normalizado: minúsculas, sin espacios al borde).
cat_cols = ["wilderness_area", "soil_type"]
checks["categorias_normalizadas"] = bool(
    all(gold_df[c].astype(str).str.strip().str.lower().equals(gold_df[c].astype(str)) for c in cat_cols)
)

import pprint
pprint.pprint(checks)
assert all(checks.values()), f"Gold Layer no supera validación: {checks}"
print("\nGold Layer validada ✔")

## 08. Definición del problema

La variable objetivo **no se asume**: se decide tras inspeccionar el esquema. `cover_type` es la
etiqueta natural del dataset Covertype (el tipo de cubierta forestal de cada parcela de 30×30 m);
las demás columnas son atributos cartográficos/topográficos observados *antes* de conocer la
cubierta, por lo que son predictores legítimos.

- **Variable objetivo:** `cover_type` (0–6 en esta Data API; 7 clases).
- **Tipo de problema:** **clasificación multiclase** (7 clases, enteras, sin orden natural fuerte).
- **Predictores:** 10 numéricas (elevación, pendiente, aspecto, distancias, hillshade) + 2
  categóricas (`wilderness_area`, `soil_type`).
- **Descartadas:** columnas de linaje/identificador (`id`, `raw_id`, `processed_id`,
  `dataset_version`, `dag_run_id`, timestamps) — identificadores que generan *leakage* o ruido.
- **Variables candidatas a target alternativas:** no hay otra razonable; el resto son features.

**Nota geográfica / leakage espacial:** no hay latitud/longitud, pero `wilderness_area` es una
variable de localización. Es un predictor válido en el benchmark estándar de Covertype; aun así,
las parcelas vecinas están autocorrelacionadas espacialmente, por lo que un *split aleatorio*
puede filtrar información entre celdas contiguas. El split del proyecto (congelado en la base) es
aleatorio; se documenta como alternativa más conservadora una **validación espacial por grupos**
(`GroupKFold`/split por `wilderness_area`). Para mantener compatibilidad con el split fijo de la
arquitectura, aquí se usa el split de la base y se deja anotada la alternativa.

In [ ]:
# Distribución del target y chequeo de desbalance.
target_counts = gold_df[TARGET].value_counts().sort_index()
target_pct = (target_counts / len(gold_df) * 100).round(2)
dist = pd.DataFrame({"n": target_counts, "pct": target_pct})
display(dist)

imbalance_ratio = target_counts.max() / target_counts.min()
print(f"clases: {gold_df[TARGET].nunique()} | ratio desbalance (max/min): {imbalance_ratio:.1f}x")
PROBLEM_TYPE = "multiclass"
print("tipo de problema:", PROBLEM_TYPE)

## 09. Análisis exploratorio (EDA)

EDA acotado y reproducible. Los gráficos se intentan con matplotlib/seaborn; si no están
disponibles, el notebook sigue mostrando los resúmenes numéricos (no falla en entorno headless).

In [ ]:
NUMERIC = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
CATEGORICAL = ["wilderness_area", "soil_type"]

print("resumen numérico:")
display(gold_df[NUMERIC].describe().T)

print("\ncardinalidad categórica:")
for c in CATEGORICAL:
    print(f"  {c}: {gold_df[c].nunique()} categorías")

In [ ]:
# Correlaciones entre numéricas (detecta redundancia; ayuda a razonar sobre hillshade).
corr = gold_df[NUMERIC].corr(numeric_only=True).round(2)
display(corr)

try:
    import matplotlib
    matplotlib.use("Agg")  # backend no interactivo: seguro en headless
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.heatmap(corr, annot=False, cmap="viridis", ax=axes[0])
    axes[0].set_title("Correlación (numéricas)")
    gold_df[TARGET].value_counts().sort_index().plot(kind="bar", ax=axes[1])
    axes[1].set_title("Distribución del target (cover_type)")
    plt.tight_layout()
    fig.savefig(ARTIFACTS_DIR / "eda_overview.png", dpi=90)
    plt.show()
    print("figura guardada en", ARTIFACTS_DIR / "eda_overview.png")
except Exception as exc:  # noqa: BLE001
    print(f"(gráficos omitidos: {exc})")

## 10. Feature Engineering

Transformaciones **fila a fila y deterministas** (sin aprender parámetros de los datos → sin
leakage, aplicables de forma idéntica a train/validation/test):

1. **Tipado correcto:** numéricas a `int/float`, categóricas a `category`.
2. **Feature derivada justificada:** `euclidean_distance_to_hydrology` =
   √(horizontal² + vertical²) — distancia en línea recta al agua, combinación natural de dos
   componentes ya presentes.

La **codificación de categóricas** (`wilderness_area`, `soil_type`) **no** se hace a mano: se
delega al preprocesamiento interno de AutoGluon, de modo que la misma transformación viaje con el
modelo (clave para la futura *inference API*). `FEATURE_ENGINEERING` permite desactivar la feature
derivada para alinear exactamente con el contrato de la base si el equipo lo prefiere.

In [ ]:
FEATURE_ENGINEERING = True  # False -> usar exactamente el contrato de features de la base


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    '''Transformación determinista fila a fila (sin fit). Devuelve copia.'''
    out = df.copy()
    # Tipado
    for c in NUMERIC:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    for c in CATEGORICAL:
        out[c] = out[c].astype("category")
    out[TARGET] = out[TARGET].astype(int)
    # Feature derivada (sin leakage)
    if FEATURE_ENGINEERING:
        out["euclidean_distance_to_hydrology"] = np.sqrt(
            out["horizontal_distance_to_hydrology"].astype(float) ** 2
            + out["vertical_distance_to_hydrology"].astype(float) ** 2
        ).round(3)
    return out


gold_fe = engineer(gold_df)

FEATURES = NUMERIC + CATEGORICAL + (
    ["euclidean_distance_to_hydrology"] if FEATURE_ENGINEERING else []
)
print(f"{len(FEATURES)} features:", FEATURES)
gold_fe.head()

## 11. Train / Validation / Test

Se respeta el split **ya decidido** (en la base para un `dataset_version`, o determinista en
memoria en el fallback). **El conjunto TEST permanece aislado**: no se usa en entrenamiento, ni en
la comparación de las 25 configuraciones, ni en la selección. Sólo se evalúa una vez, al final,
sobre la configuración ganadora (sección 15/18).

In [ ]:
def split_xy(df, split):
    part = df[df["split"] == split]
    X = part[FEATURES].reset_index(drop=True)
    y = part[TARGET].reset_index(drop=True)
    return X, y


X_train, y_train = split_xy(gold_fe, "train")
X_val, y_val = split_xy(gold_fe, "validation")
X_test, y_test = split_xy(gold_fe, "test")

# AutoGluon trabaja con un DataFrame que incluye la etiqueta.
train_data = X_train.copy(); train_data[TARGET] = y_train
val_data = X_val.copy(); val_data[TARGET] = y_val

DATASET_SIZES = {"train": len(X_train), "validation": len(X_val), "test": len(X_test)}
N_FEATURES = len(FEATURES)
CLASSES = sorted(gold_fe[TARGET].unique().tolist())
print("tamaños:", DATASET_SIZES, "| features:", N_FEATURES, "| clases:", CLASSES)

# Guard: TEST no debe solaparse con TRAIN (chequeo de integridad del split).
assert DATASET_SIZES["test"] > 0 and DATASET_SIZES["train"] > 0
print("split verificado — TEST aislado ✔")

## 12. Definición del modelo

**Una sola familia de modelo:** árboles con *gradient boosting* (**LightGBM**, clave `GBM` en
AutoGluon). **No es un benchmark entre algoritmos.** Justificación:

- **Naturaleza del problema:** clasificación multiclase tabular → los *gradient-boosted trees* son
  el estado del arte práctico.
- **Tipo de variables:** mezcla numéricas + categóricas; LightGBM (vía AutoGluon) maneja ambas y
  relaciones no lineales sin escalado.
- **Tamaño del dataset:** Covertype es grande (cientos de miles de filas); LightGBM escala y
  entrena rápido.
- **Interpretabilidad:** ofrece importancias de features y es más inspeccionable que una red.
- **Tiempo de entrenamiento:** rápido → factible correr 25 experimentos.
- **Compatibilidad con MLflow:** el `TabularPredictor` serializa a carpeta; `mlflow.log_model`
  puede registrarlo luego sin fricción.

**Familia intercambiable:** cambiar `MODEL_FAMILY` (p. ej. `"XGB"`, `"CAT"`, `"RF"`) y el espacio
de búsqueda re-apunta a esa familia sin tocar el resto del flujo.

In [ ]:
# Familia única. AutoGluon keys: 'GBM' (LightGBM), 'XGB', 'CAT' (CatBoost), 'RF', 'XT', 'NN_TORCH'...
MODEL_FAMILY = "GBM"

# Métrica que AutoGluon usa internamente para ordenar/entrenar (coherente con la principal).
AG_EVAL_METRIC = "f1_macro"
print("familia de modelo:", MODEL_FAMILY, "| eval_metric AutoGluon:", AG_EVAL_METRIC)

## 13. Diseño de las 25 configuraciones

Espacio de búsqueda **técnicamente justificado** para LightGBM, no 25 variaciones mínimas:
**5 tasas de aprendizaje × 5 niveles de capacidad/regularización = 25** configuraciones que
barren zonas distintas (modelos pequeños y muy regularizados → modelos grandes y poco
regularizados).

Cada configuración incluye `config_id`, `experiment`, `model`, `hyperparameters`, `seed` y
`timestamp`. Se usa HPO **en malla determinista** (no la búsqueda interna de AutoGluon) porque el
objetivo académico es tener **25 experimentos independientes claramente identificables y
reproducibles** — justamente el caso donde la documentación de AutoGluon sugiere que su HPO no es
lo ideal frente a configuraciones explícitas.

In [ ]:
LEARNING_RATES = [0.01, 0.03, 0.05, 0.1, 0.2]

# 5 niveles: capacidad baja + regularización fuerte  ->  capacidad alta + regularización débil.
CAPACITY_TIERS = [
    {"num_leaves": 15,  "max_depth": 4,  "num_boost_round": 200,  "min_data_in_leaf": 60,
     "feature_fraction": 0.60, "bagging_fraction": 0.60, "bagging_freq": 1, "lambda_l1": 1.0, "lambda_l2": 1.0},
    {"num_leaves": 31,  "max_depth": 6,  "num_boost_round": 400,  "min_data_in_leaf": 40,
     "feature_fraction": 0.70, "bagging_fraction": 0.70, "bagging_freq": 1, "lambda_l1": 0.5, "lambda_l2": 0.5},
    {"num_leaves": 63,  "max_depth": 8,  "num_boost_round": 600,  "min_data_in_leaf": 20,
     "feature_fraction": 0.80, "bagging_fraction": 0.80, "bagging_freq": 1, "lambda_l1": 0.1, "lambda_l2": 0.1},
    {"num_leaves": 127, "max_depth": 12, "num_boost_round": 800,  "min_data_in_leaf": 10,
     "feature_fraction": 0.90, "bagging_fraction": 0.90, "bagging_freq": 1, "lambda_l1": 0.0, "lambda_l2": 0.05},
    {"num_leaves": 255, "max_depth": -1, "num_boost_round": 1200, "min_data_in_leaf": 5,
     "feature_fraction": 1.00, "bagging_fraction": 1.00, "bagging_freq": 0, "lambda_l1": 0.0, "lambda_l2": 0.0},
]


def build_configurations(seed=SEED, model_family=MODEL_FAMILY):
    configs, idx = [], 1
    stamp = datetime.now(timezone.utc).isoformat()
    for lr in LEARNING_RATES:
        for tier in CAPACITY_TIERS:
            hp = {"learning_rate": lr, "seed": seed, **tier}
            configs.append({
                "config_id": f"config_{idx:02d}",
                "experiment": f"Experiment {idx:02d}",
                "model": model_family,
                "hyperparameters": hp,
                "seed": seed,
                "timestamp": stamp,
            })
            idx += 1
    return configs


configurations = build_configurations()
assert len(configurations) == N_CONFIGS, f"se esperaban {N_CONFIGS}, hay {len(configurations)}"
_sigs = {json.dumps(c["hyperparameters"], sort_keys=True) for c in configurations}
assert len(_sigs) == N_CONFIGS, "las 25 configuraciones deben ser distintas"

# Persistir el espacio de configuraciones (trazabilidad / reproducibilidad).
with open(ARTIFACTS_DIR / "configurations.json", "w") as fh:
    json.dump(configurations, fh, indent=2)

print(f"{len(configurations)} configuraciones únicas generadas")
pd.json_normalize(configurations).head()

## 14. Ejecución de los 25 experimentos

Bucle sobre las 25 configuraciones. Cada una entrena un `TabularPredictor` **independiente** con
**un solo modelo GBM** (sin *bagging*, sin *stacking*, sin *weighted ensemble*) para que la
configuración quede perfectamente aislada y reproducible. Se guarda en `models/config_XX/` y se
registran parámetros, métricas de validación, tiempo de entrenamiento, tamaños, nº de features y
semilla. **El TEST no se toca aquí.**

La estructura del bucle es exactamente la que MLflow consumirá después (sección 20):
*entrenar → calcular métricas → guardar modelo → guardar params → guardar métricas → registrar
resultado*.

In [ ]:
def compute_metrics(y_true, y_pred, y_proba=None, labels=None, prefix=""):
    '''Métricas de clasificación multiclase, con nombres MLflow-friendly.'''
    m = {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        f"{prefix}precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_proba is not None:
        try:
            m[f"{prefix}log_loss"] = log_loss(y_true, y_proba, labels=labels)
            m[f"{prefix}roc_auc_ovr_macro"] = roc_auc_score(
                y_true, y_proba, multi_class="ovr", average="macro", labels=labels
            )
        except Exception:  # noqa: BLE001
            pass
    return {k: float(round(v, 6)) for k, v in m.items()}

In [ ]:
PRIMARY_METRIC = "val_f1_macro"   # métrica principal de comparación (ver sección 8 y 15)
QUICK_SMOKE = False               # True = subconjunto pequeño para probar el flujo rápido

_train = train_data.sample(n=min(20000, len(train_data)), random_state=SEED) if QUICK_SMOKE else train_data

records = []
for cfg in configurations:
    cid = cfg["config_id"]
    model_path = MODELS_DIR / cid
    t0 = time.perf_counter()

    predictor = TabularPredictor(
        label=TARGET,
        problem_type=PROBLEM_TYPE,
        eval_metric=AG_EVAL_METRIC,
        path=str(model_path),
        verbosity=0,
    )
    predictor.fit(
        train_data=_train,
        hyperparameters={MODEL_FAMILY: cfg["hyperparameters"]},
        num_bag_folds=0,
        num_stack_levels=0,
        fit_weighted_ensemble=False,
    )
    train_time = round(time.perf_counter() - t0, 3)

    # Evaluación en VALIDATION (nunca en test aquí).
    val_pred = predictor.predict(X_val)
    try:
        val_proba = predictor.predict_proba(X_val)[CLASSES].to_numpy()
    except Exception:  # noqa: BLE001
        val_proba = None
    val_metrics = compute_metrics(y_val, val_pred, val_proba, labels=CLASSES, prefix="val_")

    record = {
        "config_id": cid,
        "experiment": cfg["experiment"],
        "model": cfg["model"],
        "seed": cfg["seed"],
        "timestamp": cfg["timestamp"],
        "train_time_s": train_time,
        "n_train": DATASET_SIZES["train"],
        "n_validation": DATASET_SIZES["validation"],
        "n_test": DATASET_SIZES["test"],
        "n_features": N_FEATURES,
        "dataset_version": DATASET_VERSION,
        "model_path": str(model_path),
        **{f"param_{k}": v for k, v in cfg["hyperparameters"].items()},
        **val_metrics,
    }
    records.append(record)

    # Payload por-run listo para MLflow (params + metrics + artefactos).
    with open(MLFLOW_DIR / f"{cid}.json", "w") as fh:
        json.dump({
            "run_name": cfg["experiment"],
            "params": {**cfg["hyperparameters"], "model_family": cfg["model"],
                       "dataset_version": DATASET_VERSION, "n_features": N_FEATURES},
            "metrics": val_metrics,
            "artifacts": {"model_dir": str(model_path),
                          "config_json": str(ARTIFACTS_DIR / "configurations.json")},
            "tags": {"seed": cfg["seed"], "problem_type": PROBLEM_TYPE},
        }, fh, indent=2)

    print(f"{cfg['experiment']:<14} {PRIMARY_METRIC}={record[PRIMARY_METRIC]:.4f}  ({train_time}s)")

print("\n25 experimentos completados")

## 15. Evaluación

**Métrica principal:** `f1_macro` en validation. Justificación: Covertype está **desbalanceado**
(clases mayoritarias dominan); *accuracy* premiaría al modelo por acertar las clases grandes e
ignoraría las pequeñas. `f1_macro` pondera por igual las 7 clases, penalizando el mal desempeño en
minoritarias. Se reportan además *accuracy*, *balanced_accuracy*, precisión/recall macro,
*log_loss* y ROC-AUC (OVR macro) como contexto, pero la decisión se toma con `f1_macro`.

La **matriz de confusión** se calcula sólo para la configuración ganadora, en validation.

In [ ]:
results_df = pd.DataFrame(records)

best_row = results_df.sort_values(PRIMARY_METRIC, ascending=False).iloc[0]
BEST_CONFIG_ID = best_row["config_id"]
print(f"mejor configuración por {PRIMARY_METRIC}: {BEST_CONFIG_ID} = {best_row[PRIMARY_METRIC]:.4f}")

# Matriz de confusión (validation) de la ganadora.
best_predictor = TabularPredictor.load(str(MODELS_DIR / BEST_CONFIG_ID))
val_pred_best = best_predictor.predict(X_val)
cm = confusion_matrix(y_val, val_pred_best, labels=CLASSES)
display(pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                     columns=[f"pred_{c}" for c in CLASSES]))
print(classification_report(y_val, val_pred_best, zero_division=0))

## 16. Consolidación de resultados

`results_df`: **exactamente 25 filas** (una por configuración), con `config_id`, `model`, los
hiperparámetros (`param_*`), las métricas (`val_*`) y metadatos del experimento.

In [ ]:
assert len(results_df) == N_CONFIGS, f"results_df debe tener {N_CONFIGS} filas, tiene {len(results_df)}"

col_order = (
    ["config_id", "experiment", "model", "dataset_version",
     "n_train", "n_validation", "n_test", "n_features", "seed", "train_time_s"]
    + [c for c in results_df.columns if c.startswith("param_")]
    + [c for c in results_df.columns if c.startswith("val_")]
    + ["model_path", "timestamp"]
)
results_df = results_df[[c for c in col_order if c in results_df.columns]]
print("results_df:", results_df.shape)
results_df.head()

## 17. Comparación de las 25 configuraciones

Ranking por la métrica principal y lectura rápida del efecto de `learning_rate` y capacidad.

In [ ]:
ranking = results_df.sort_values(PRIMARY_METRIC, ascending=False)[
    ["config_id", "param_learning_rate", "param_num_leaves", "param_num_boost_round",
     PRIMARY_METRIC, "val_accuracy", "val_balanced_accuracy", "train_time_s"]
].reset_index(drop=True)
display(ranking)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(results_df["config_id"], results_df[PRIMARY_METRIC])
    ax.set_xticklabels(results_df["config_id"], rotation=90)
    ax.set_ylabel(PRIMARY_METRIC); ax.set_title("25 configuraciones · " + PRIMARY_METRIC)
    plt.tight_layout(); fig.savefig(ARTIFACTS_DIR / "comparison.png", dpi=90); plt.show()
except Exception as exc:  # noqa: BLE001
    print(f"(gráfico omitido: {exc})")

## 18. Selección del modelo/configuración para la siguiente etapa

Se elige la configuración con mejor `f1_macro` en validation. **Sólo ahora** se evalúa el
**TEST**, una única vez, para reportar una estimación honesta del desempeño del modelo elegido
(el test no influyó en la selección).

In [ ]:
test_pred = best_predictor.predict(X_test)
try:
    test_proba = best_predictor.predict_proba(X_test)[CLASSES].to_numpy()
except Exception:  # noqa: BLE001
    test_proba = None
test_metrics = compute_metrics(y_test, test_pred, test_proba, labels=CLASSES, prefix="test_")

selection = {
    "selected_config_id": BEST_CONFIG_ID,
    "primary_metric": PRIMARY_METRIC,
    "primary_metric_value": float(best_row[PRIMARY_METRIC]),
    "model_family": MODEL_FAMILY,
    "model_path": str(MODELS_DIR / BEST_CONFIG_ID),
    "dataset_version": DATASET_VERSION,
    "test_metrics": test_metrics,
}
with open(ARTIFACTS_DIR / "selected_model.json", "w") as fh:
    json.dump(selection, fh, indent=2)

print(json.dumps(selection, indent=2))

## 19. Exportación de resultados

Tabla consolidada a `results.csv` (siempre) y `results.parquet` (si hay motor parquet). Formatos
que MLflow u otra herramienta pueden consumir directamente.

In [ ]:
csv_path = RESULTS_DIR / "results.csv"
results_df.to_csv(csv_path, index=False)
print("guardado:", csv_path)

try:
    parquet_path = RESULTS_DIR / "results.parquet"
    results_df.to_parquet(parquet_path, index=False)
    print("guardado:", parquet_path)
except Exception as exc:  # noqa: BLE001
    print(f"(parquet omitido — instala pyarrow: {exc})")

# Verificación final de la estructura de artefactos.
print("\nárbol de artefactos:")
for p in sorted(ARTIFACTS_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(ARTIFACTS_DIR))

## 20. Preparación para integración con MLflow

MLflow **no** se implementa aquí. Pero cada experimento ya dejó, en `artifacts/mlflow_payloads/`,
un JSON con `params`, `metrics`, `tags` y rutas de artefactos; y cada modelo quedó en
`models/config_XX/`. El bucle conceptual que otro componente añadirá es un reemplazo 1:1 del de la
sección 14:

```python
import mlflow

mlflow.set_experiment("cubierta-forestal")
for cfg, payload in zip(configurations, payloads):     # payloads = *.json ya generados
    with mlflow.start_run(run_name=payload["run_name"]):
        mlflow.log_params(payload["params"])           # hiperparámetros + metadata
        mlflow.log_metrics(payload["metrics"])          # val_f1_macro, val_accuracy, ...
        mlflow.log_artifact(payload["artifacts"]["config_json"])
        mlflow.log_artifacts(payload["artifacts"]["model_dir"], artifact_path="model")
        # o, para el formato nativo, un flavor/pyfunc que envuelva el TabularPredictor:
        # mlflow.pyfunc.log_model(artifact_path="model", python_model=AutogluonWrapper(...))
```

La celda siguiente muestra el *payload* ya listo para un run, sin ejecutar MLflow.

In [ ]:
example = json.load(open(MLFLOW_DIR / f"{BEST_CONFIG_ID}.json"))
print("payload de ejemplo (config ganadora), listo para mlflow.log_*:\n")
print(json.dumps(example, indent=2))

## ENVIRONMENT

Versiones utilizadas, para reproducibilidad.

In [ ]:
import platform

env_info = {"python": platform.python_version(), "seed": SEED, "dataset_version": DATASET_VERSION}
for mod in ["numpy", "pandas", "sklearn", "sqlalchemy", "autogluon", "lightgbm", "psycopg", "pyarrow"]:
    try:
        m = __import__(mod)
        env_info[mod] = getattr(m, "__version__", "?")
    except Exception:  # noqa: BLE001
        env_info[mod] = "no instalado"

with open(ARTIFACTS_DIR / "environment.json", "w") as fh:
    json.dump(env_info, fh, indent=2)
print(json.dumps(env_info, indent=2))